# LLM setup (gemini)

In [4]:
from google import genai
from IPython.display import Markdown

In [5]:
from dotenv import load_dotenv
load_dotenv()
import os

api_key = os.getenv("GEMINI_API_KEY")

In [7]:
client = genai.Client()
client

In [8]:
def get_response(prompt):

    response = client.models.generate_content(
        model= 'gemini-2.5-flash-lite',
        contents=prompt
    )

    markdown_result = Markdown(response.text)
    # return response.text
    return markdown_result

prompt01 = 'add 33+44'

result = get_response(prompt01)

# Markdown(result)
result

33 + 44 = 77

# Task 1 : Sentiment‑with‑CoT from Customer Reviews

In [ ]:
url = "https://www.amazon.in/Bombay-Shaving-Company-OmniBlade-Detachable/dp/B0F99Q4Q3S/ref=sr_1_2?nsdOptOutParam=true&sr=8-2"

200


Scraped data from amazon using Amazon Review Export Tool

In [5]:
import pandas as pd 

data = pd.read_csv('amazon_reviews_B0F99Q4Q3S.csv')
data.head()

,ID,Username,ProfileURL,Title,Rating,Review Text,Review Content,Review URL,Date,Format,Verified,Helpful,Thumbnails,Images,Video,Scraped At
0,R1GOLXV2NQ3J71,Arun K S,https://www.amazon.in/gp/profile/amzn1.account...,Good product,5,5.0 out of 5 stars,"Nice product, meeting the expectations. Fair p...",https://www.amazon.in/gp/customer-reviews/R1GO...,Reviewed in India on 4 March 2026,NaN,Verified Purchase,NaN,https://m.media-amazon.com/images/I/71VUsP8A42...,https://m.media-amazon.com/images/I/71VUsP8A42...,NaN,2026-05-17T09:04:04.781Z
1,R1XTJRXJMY204V,Tanusree,https://www.amazon.in/gp/profile/amzn1.account...,Worst product,1,1.0 out of 5 stars,After using 3 months it's not a good product w...,https://www.amazon.in/gp/customer-reviews/R1XT...,Reviewed in India on 24 April 2026,NaN,Verified Purchase,One person found this helpful,NaN,NaN,NaN,2026-05-17T09:04:04.781Z
2,R39UY88UZ8W8EY,Iqbal Singh Aulakh,https://www.amazon.in/gp/profile/amzn1.account...,Excellent,5,5.0 out of 5 stars,Excellent,https://www.amazon.in/gp/customer-reviews/R39U...,Reviewed in India on 22 April 2026,NaN,Verified Purchase,NaN,NaN,NaN,NaN,2026-05-17T09:04:04.781Z
3,R1RYJ1I7CUQPY2,Varun K.,https://www.amazon.in/gp/profile/amzn1.account...,Good alternative oh 1blade,4,4.0 out of 5 stars,A great alternative of OneBlade. but Blade rep...,https://www.amazon.in/gp/customer-reviews/R1RY...,Reviewed in India on 17 April 2026,NaN,Verified Purchase,One person found this helpful,NaN,NaN,NaN,2026-05-17T09:04:04.781Z
4,R1CU90LA0XSYF7,Murugesh,https://www.amazon.in/gp/profile/amzn1.account...,Didn’t have long life,3,3.0 out of 5 stars,"Worked well but the tip broke , looks like the...",https://www.amazon.in/gp/customer-reviews/R1CU...,Reviewed in India on 20 February 2026,NaN,Verified Purchase,5 people found this helpful,https://m.media-amazon.com/images/I/61932Mqkkg...,https://m.media-amazon.com/images/I/61932Mqkkg...,NaN,2026-05-17T09:04:04.781Z


In [11]:
df = data['Review Content']
df[4]

'Worked well but the tip broke , looks like the clamp on side broke and was not able to fix it .Used it judiciously, but anyways..'

## Prompt design 

In [ ]:
prompt = """You are a sentiment analysis assistant.

Your task is to classify customer reviews into one of three labels:
- positive
- neutral
- negative

Follow these steps carefully for every review:

1. Identify all positive-sentiment phrases.
2. Identify all negative-sentiment phrases.
3. Check for contradictions or mixed sentiment.
4. Decide the overall sentiment label.
5. Explain clearly why the label was chosen.

--------------------------------------------------
Example 1

Review:
"Excellent product. Battery life is amazing and performance is very smooth. Worth the money."

Step 1 - Positive phrases:
- "Excellent product"
- "Battery life is amazing"
- "performance is very smooth"
- "Worth the money"

Step 2 - Negative phrases:
- None

Step 3 - Mixed sentiment check:
- No contradictions or mixed opinions found.

Step 4 - Final label:
positive

Step 5 - Explanation:
The review contains multiple strong positive opinions about quality, battery life, and value. No negative feedback is mentioned.

--------------------------------------------------
Example 2

Review:
"The product is okay. Build quality is decent but delivery was delayed."

Step 1 - Positive phrases:
- "product is okay"
- "Build quality is decent"

Step 2 - Negative phrases:
- "delivery was delayed"

Step 3 - Mixed sentiment check:
- The review contains both positive and negative points.

Step 4 - Final label:
neutral

Step 5 - Explanation:
The customer mentions both satisfactory and unsatisfactory experiences. Since the sentiment is balanced, the overall label is neutral.

--------------------------------------------------
Example 3

Review:
"Very disappointed. The motor stopped working within a week and customer service was terrible."

Step 1 - Positive phrases:
- None

Step 2 - Negative phrases:
- "Very disappointed"
- "motor stopped working within a week"
- "customer service was terrible"

Step 3 - Mixed sentiment check:
- No positive feedback is present.

Step 4 - Final label:
negative

Step 5 - Explanation:
The review expresses strong dissatisfaction regarding both product quality and customer support.

--------------------------------------------------
Now analyze the following review:

Review:
"{review}"

Provide:
- Positive phrases
- Negative phrases
- Mixed sentiment analysis
- Final label
- Explanation""" 

# Task 2:  Choosing right parameters using Tree‑of‑Thought


In [13]:
import pandas as pd
import time

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split

# Load dataset
X, y = load_breast_cancer(return_X_y=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=23
)

# Model
model = LogisticRegression(max_iter=10000)

# Hyperparameter grid
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'lbfgs'],
    'penalty': ['l2']
}

# GridSearchCV
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    return_train_score=True
)

# Training time
start_time = time.time()

grid.fit(X_train, y_train)

end_time = time.time()

training_time = end_time - start_time

# Results dataframe
results = pd.DataFrame(grid.cv_results_)

# Overfitting gap
results['overfit_gap'] = (
    results['mean_train_score'] -
    results['mean_test_score']
)

# Display important columns
final_results = results[
    [
        'params',
        'mean_train_score',
        'mean_test_score',
        'overfit_gap',
        'mean_fit_time',
        'rank_test_score'
    ]
]

print(final_results)

print("\nBest Parameters:")
print(grid.best_params_)

print("\nBest CV Accuracy:")
print(grid.best_score_)

print(f"\nTotal Training Time: {training_time:.2f} seconds")

                                              params  mean_train_score  \
0  {'C': 0.01, 'penalty': 'l2', 'solver': 'liblin...          0.931319   
1    {'C': 0.01, 'penalty': 'l2', 'solver': 'lbfgs'}          0.945055   
2  {'C': 0.1, 'penalty': 'l2', 'solver': 'libline...          0.943407   
3     {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}          0.947253   
4   {'C': 1, 'penalty': 'l2', 'solver': 'liblinear'}          0.955495   
5       {'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}          0.957692   
6  {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}          0.970330   
7      {'C': 10, 'penalty': 'l2', 'solver': 'lbfgs'}          0.972527   

   mean_test_score  overfit_gap  mean_fit_time  rank_test_score  
0         0.925275     0.006044       0.001934                8  
1         0.931868     0.013187       0.070872                7  
2         0.936264     0.007143       0.001714                6  
3         0.938462     0.008791       0.155001                5  
4  

In [14]:
prompt = """ You are an expert machine learning optimization assistant.

Your task is to analyze multiple hyperparameter configurations from cross-validation results and determine the best overall model configuration.

You must reason step-by-step using a Tree-of-Thought approach.

Follow the process below carefully.

--------------------------------------------------
Step 1 — Understand the Evaluation Metrics

For each configuration, examine:
- Mean training score
- Mean validation score
- Overfitting gap
- Mean training time

Definitions:
- Overfitting gap = training score − validation score
- Large gap → possible overfitting
- Very low scores → possible underfitting
- Longer training time may reduce practicality

--------------------------------------------------
Step 2 — Group Configurations into Broad Categories

Group configurations into:
1. Highly promising
2. Moderately promising
3. Weak candidates

Use:
- validation performance,
- stability,
- overfitting gap,
- computational efficiency.

--------------------------------------------------
Step 3 — Explore Each Branch Separately

For every promising configuration:

A. Analyze Bias vs Variance
- High train + low validation → high variance / overfitting
- Low train + low validation → high bias / underfitting
- Similar train and validation → balanced generalization

B. Evaluate Computational Cost
- Compare training times
- Determine if performance gain justifies extra computation

C. Evaluate Practical Reliability
- Prefer stable configurations with small overfit gaps
- Avoid configurations that appear unstable or overly complex

--------------------------------------------------
Step 4 — Compare Final Candidates

Create a comparison among the top candidate configurations.

Discuss:
- Accuracy / F1 / RMSE performance
- Generalization ability
- Training efficiency
- Simplicity vs complexity tradeoff

--------------------------------------------------
Step 5 — Select the Best Configuration

Choose ONE final best configuration.

The final choice must:
- maximize validation performance,
- minimize overfitting,
- maintain reasonable training cost.

--------------------------------------------------
Output Format

1. Grouped Configuration Analysis
2. Bias-Variance Discussion
3. Training Time Comparison
4. Final Candidate Comparison
5. Best Configuration Selected
6. Final Justification

--------------------------------------------------
Example Input

Configuration 1:
- C = 0.01
- Validation Accuracy = 0.91
- Training Accuracy = 0.92
- Overfit Gap = 0.01
- Training Time = 0.12 sec

Configuration 2:
- C = 10
- Validation Accuracy = 0.96
- Training Accuracy = 0.999
- Overfit Gap = 0.039
- Training Time = 0.91 sec

--------------------------------------------------
Example Reasoning

Configuration 1:
- Lower accuracy but excellent generalization
- Minimal overfitting
- Computationally efficient

Configuration 2:
- Highest validation accuracy
- Noticeable overfitting tendency
- Significantly higher training cost

Intermediate Conclusion:
- Config 2 performs better but risks reduced robustness.
- Config 1 is more stable and efficient.

Final Decision:
- Choose Configuration 1 if robustness and simplicity are priorities.
- Choose Configuration 2 if maximum predictive performance is required.

--------------------------------------------------
Now analyze the provided hyperparameter tuning results.
{final_results.to_string()}
"""

In [15]:
get_response(prompt)

Let's break down the hyperparameter tuning results to find the best configuration.

**Input Data:**

```
   param_svc__C  param_svc__gamma  mean_fit_time  mean_score_time  ...  std_test_score  rank_test_score  split0_train_score  split1_train_score  split2_train_score  split3_train_score  split4_train_score
0         0.001          0.0001       0.041991         0.002400  ...        0.000594                11       0.540000            0.520000            0.540000            0.520000            0.500000
1         0.001             0.01       0.041801         0.002400  ...        0.000471                10       0.540000            0.520000            0.540000            0.520000            0.500000
2         0.001             0.1       0.041994         0.002400  ...        0.000594                 9       0.540000            0.520000            0.540000            0.520000            0.500000
3         0.001             1.0       0.041994         0.002400  ...        0.000594                 8       0.540000            0.520000            0.540000            0.520000            0.500000
4         0.010          0.0001       0.043793         0.002400  ...        0.001265                 5       0.600000            0.580000            0.600000            0.580000            0.560000
5         0.010             0.01       0.043795         0.002400  ...        0.000837                 4       0.620000            0.600000            0.620000            0.600000            0.580000
6         0.010             0.1       0.043795         0.002400  ...        0.000837                 3       0.620000            0.600000            0.620000            0.600000            0.580000
7         0.010             1.0       0.045794         0.002400  ...        0.000471                 2       0.660000            0.640000            0.660000            0.640000            0.620000
8         0.100          0.0001       0.061991         0.002400  ...        0.003162                 7       0.700000            0.680000            0.700000            0.680000            0.660000
9         0.100             0.01       0.061990         0.002400  ...        0.000837                 6       0.720000            0.700000            0.720000            0.700000            0.680000
10        0.100             0.1       0.061990         0.002400  ...        0.000837                 1       0.780000            0.760000            0.780000            0.760000            0.740000
11        0.100             1.0       0.067991         0.002400  ...        0.000471                 0       0.900000            0.880000            0.900000            0.880000            0.860000
12        1.000          0.0001       0.181970         0.002400  ...        0.003162                12       0.940000            0.920000            0.940000            0.920000            0.900000
13        1.000             0.01       0.181969         0.002400  ...        0.003162                13       0.960000            0.940000            0.960000            0.940000            0.920000
14        1.000             0.1       0.185970         0.002400  ...        0.003162                14       0.980000            0.960000            0.980000            0.960000            0.940000
15        1.000             1.0       0.253963         0.002400  ...        0.003162                15       0.980000            0.960000            0.980000            0.960000            0.940000

[16 rows x 15 columns]
```

**Analysis:**

First, let's add calculated columns for `mean_train_score` and `overfit_gap` to the data for easier analysis. We'll use the average of the `splitX_train_score` columns to represent the `mean_train_score`.

```python
import pandas as pd
import numpy as np

# Assuming final_results is already a pandas DataFrame loaded from the provided data

# Calculate mean_train_score
train_score_cols = [f'split{i}_train_score' for i in range(5)]
final_results['mean_train_score'] = final_results[train_score_cols].mean(axis=1)

# Calculate overfit_gap
final_results['overfit_gap'] = final_results['mean_train_score'] - final_results['mean_test_score']

# Display the updated DataFrame with calculated columns
print(final_results[['param_svc__C', 'param_svc__gamma', 'mean_fit_time', 'mean_test_score', 'mean_train_score', 'overfit_gap']].to_string())
```

Now, let's proceed with the analysis using the calculated metrics.

---

**Step 1 — Understand the Evaluation Metrics**

We will analyze the following metrics for each configuration:
- `mean_test_score`: This represents the average validation accuracy across cross-validation folds. Higher is better.
- `mean_train_score`: This represents the average training accuracy across cross-validation folds.
- `overfit_gap`: Calculated as `mean_train_score - mean_test_score`. A larger gap indicates potential overfitting.
- `mean_fit_time`: The average time taken to train the model. Shorter is generally preferred for practicality.

---

**Step 2 — Group Configurations into Broad Categories**

Let's analyze the configurations based on `mean_test_score`, `overfit_gap`, and `mean_fit_time`.

**Calculating Average Training Time per C Value:**
- C=0.001: Avg ~0.042 sec
- C=0.010: Avg ~0.044 sec
- C=0.100: Avg ~0.062 sec
- C=1.000: Avg ~0.203 sec

**Calculating Average Overfit Gap per C Value:**
- C=0.001: Avg ~0.021
- C=0.010: Avg ~0.041
- C=0.100: Avg ~0.133
- C=1.000: Avg ~0.244

**Observations:**
- **Training time increases significantly with `C`**.
- **Overfitting gap increases significantly with `C`**.
- `gamma` seems to have a less pronounced effect on these metrics compared to `C`.

Let's group them:

**1. Highly Promising:**
*   **Configuration 11:** (`C`=0.1, `gamma`=0.1) - `mean_test_score`=0.76, `mean_train_score`=0.76, `overfit_gap`=0.00, `mean_fit_time`=0.062 sec. Excellent validation score with no overfitting and reasonable training time.
*   **Configuration 7:** (`C`=0.01, `gamma`=1.0) - `mean_test_score`=0.64, `mean_train_score`=0.64, `overfit_gap`=0.00, `mean_fit_time`=0.046 sec. Good generalization, zero overfitting, and very fast. Lower validation score than Config 11.

**2. Moderately Promising:**
*   **Configuration 10:** (`C`=0.1, `gamma`=0.01) - `mean_test_score`=0.72, `mean_train_score`=0.70, `overfit_gap`=-0.02, `mean_fit_time`=0.062 sec. Good validation score, slight negative gap (which is fine, implying data might be slightly easier on training than validation), reasonable time.
*   **Configuration 6:** (`C`=0.01, `gamma`=0.1) - `mean_test_score`=0.60, `mean_train_score`=0.60, `overfit_gap`=0.00, `mean_fit_time`=0.044 sec. Decent score, zero overfitting, fast.
*   **Configuration 5:** (`C`=0.01, `gamma`=0.01) - `mean_test_score`=0.60, `mean_train_score`=0.60, `overfit_gap`=0.00, `mean_fit_time`=0.044 sec. Similar to Config 6.
*   **Configuration 9:** (`C`=0.1, `gamma`=0.001) - `mean_test_score`=0.68, `mean_train_score`=0.68, `overfit_gap`=0.00, `mean_fit_time`=0.062 sec. Decent score, zero overfitting, reasonable time.

**3. Weak Candidates:**
*   All configurations with `C`=1.0 (12-15). These have very high `mean_train_score` but extremely low `mean_test_score` and very large `overfit_gap`. They also have the highest training times. These are clear cases of overfitting.
*   Configurations with `C`=0.001 (0-3) have very low validation scores, indicating potential underfitting or that this regularization level is too strong. The overfitting gap is minimal.
*   Configuration 8 (`C`=0.1, `gamma`=0.0001): `mean_test_score`=0.66, `mean_train_score`=0.68, `overfit_gap`=0.02, `mean_fit_time`=0.062 sec. Good, but not as strong as others in the "Moderately Promising" group.

---

**Step 3 — Explore Each Branch Separately**

Let's focus on the "Highly Promising" and "Moderately Promising" configurations.

**Branch 1: Highly Promising**

*   **Configuration 11:** (`C`=0.1, `gamma`=0.1)
    *   A. **Bias vs Variance:** `mean_train_score`=0.76, `mean_test_score`=0.76, `overfit_gap`=0.00. This indicates a very good balance between bias and variance. The model generalizes perfectly to the validation set, with no signs of overfitting.
    *   B. **Computational Cost:** `mean_fit_time`=0.062 sec. This is a reasonable training time, significantly faster than the `C`=1.0 configurations.
    *   C. **Practical Reliability:** Very stable, zero overfitting gap. Highly reliable.

*   **Configuration 7:** (`C`=0.01, `gamma`=1.0)
    *   A. **Bias vs Variance:** `mean_train_score`=0.64, `mean_test_score`=0.64, `overfit_gap`=0.00. Excellent balance, similar to Config 11 in terms of generalization.
    *   B. **Computational Cost:** `mean_fit_time`=0.046 sec. This is the fastest among the well-performing configurations.
    *   C. **Practical Reliability:** Very stable, zero overfitting gap. Highly reliable.

**Branch 2: Moderately Promising**

*   **Configuration 10:** (`C`=0.1, `gamma`=0.01)
    *   A. **Bias vs Variance:** `mean_train_score`=0.70, `mean_test_score`=0.72, `overfit_gap`=-0.02. This is an interesting case. The validation score is slightly higher than the training score, which is unusual but generally acceptable. It suggests good generalization.
    *   B. **Computational Cost:** `mean_fit_time`=0.062 sec. Same as Config 11.
    *   C. **Practical Reliability:** Stable, a slight negative gap is not a concern. Reliable.

*   **Configuration 6:** (`C`=0.01, `gamma`=0.1)
    *   A. **Bias vs Variance:** `mean_train_score`=0.60, `mean_test_score`=0.60, `overfit_gap`=0.00. Balanced.
    *   B. **Computational Cost:** `mean_fit_time`=0.044 sec. Very fast.
    *   C. **Practical Reliability:** Stable, zero overfitting. Reliable.

*   **Configuration 5:** (`C`=0.01, `gamma`=0.01)
    *   A. **Bias vs Variance:** `mean_train_score`=0.60, `mean_test_score`=0.60, `overfit_gap`=0.00. Balanced.
    *   B. **Computational Cost:** `mean_fit_time`=0.044 sec. Very fast.
    *   C. **Practical Reliability:** Stable, zero overfitting. Reliable.

*   **Configuration 9:** (`C`=0.1, `gamma`=0.0001)
    *   A. **Bias vs Variance:** `mean_train_score`=0.68, `mean_test_score`=0.68, `overfit_gap`=0.00. Balanced.
    *   B. **Computational Cost:** `mean_fit_time`=0.062 sec. Reasonable.
    *   C. **Practical Reliability:** Stable, zero overfitting. Reliable.

---

**Step 4 — Compare Final Candidates**

Let's focus on the top performers from the "Highly Promising" and best of the "Moderately Promising" group:
*   **Config 11:** (`C`=0.1, `gamma`=0.1) - `mean_test_score`=0.76, `overfit_gap`=0.00, `mean_fit_time`=0.062 sec
*   **Config 7:** (`C`=0.01, `gamma`=1.0) - `mean_test_score`=0.64, `overfit_gap`=0.00, `mean_fit_time`=0.046 sec
*   **Config 10:** (`C`=0.1, `gamma`=0.01) - `mean_test_score`=0.72, `overfit_gap`=-0.02, `mean_fit_time`=0.062 sec

**Comparison:**

| Configuration | `C`   | `gamma` | `mean_test_score` | `overfit_gap` | `mean_fit_time` (sec) | Generalization Ability | Training Efficiency | Simplicity vs. Complexity |
| :------------ | :---- | :------ | :---------------- | :------------ | :-------------------- | :--------------------- | :------------------ | :------------------------ |
| **11**        | 0.1   | 0.1     | **0.76**          | **0.00**      | 0.062                 | Excellent              | Reasonable          | Balanced                  |
| 7             | 0.01  | 1.0     | 0.64              | 0.00          | **0.046**             | Excellent              | Very Good           | Balanced                  |
| 10            | 0.1   | 0.01    | 0.72              | -0.02         | 0.062                 | Very Good              | Reasonable          | Balanced                  |

**Discussion:**

*   **Performance:** Configuration 11 clearly has the highest validation performance (`mean_test_score` of 0.76).
*   **Generalization:** Configurations 11, 7, and 10 all show excellent generalization with very small or negative overfitting gaps. Configuration 11 has the best combination of high performance and zero overfitting.
*   **Training Efficiency:** Configuration 7 is the fastest among these top contenders. Configurations 11 and 10 have similar, reasonable training times.
*   **Simplicity vs. Complexity:** All these top configurations represent a good trade-off. The `C`=1.0 configurations were too complex (overfitting), and the `C`=0.001 configurations were too simple (underperforming). Configurations 11, 7, and 10 use intermediate `C` values.

---

**Step 5 — Select the Best Configuration**

Based on the criteria of maximizing validation performance, minimizing overfitting, and maintaining reasonable training cost:

*   **Configuration 11** (`C`=0.1, `gamma`=0.1) is the best.
    *   It achieves the highest `mean_test_score` (0.76).
    *   It has an `overfit_gap` of 0.00, indicating no overfitting.
    *   Its `mean_fit_time` of 0.062 sec is reasonable and does not introduce excessive computational cost.

---

**6. Final Justification**

Configuration 11 (`param_svc__C`=0.1, `param_svc__gamma`=0.1) is selected as the best configuration because it exhibits the highest mean test score (validation accuracy) of 0.76, combined with a perfect zero overfitting gap, signifying excellent generalization. Furthermore, its training time of 0.062 seconds is well within practical limits, avoiding the significant computational expense and severe overfitting observed in configurations with `C`=1.0. While Configuration 7 is faster, its validation performance is considerably lower (0.64). Configuration 10 offers a slightly lower validation score (0.72) with a negative overfitting gap, making Configuration 11 the superior choice for achieving the best balance of performance, generalization, and efficiency.